# Spam Message Classifier

Walkthrough of the whole project: explore the data, clean the text, convert it to TF-IDF features, tune and train
classifiers, evaluate on unseen messages and analyse the errors. The reusable code lives in `../src/`
(`preprocess.py`, `train_model.py`, `predict.py`); this notebook is the step-by-step version.

**How to use:** run the cells from top to bottom (*Run All*). The notebook uses `../data/spam.csv`
(the SMS Spam Collection, Kaggle copy). Download it from UCI or Kaggle and put it in the `data/` folder first.
If the file is missing, the notebook falls back to the small synthetic demo file and says so. Scores on the demo
file are not realistic.

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

from preprocess import load_dataset, prepare, clean_text

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

DATA_PATH = "../data/spam.csv"            # real dataset (SMS Spam Collection)
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data/spam_dataset.csv"  # synthetic demo file
    print("spam.csv not found -> using the synthetic demo dataset. Scores on it are NOT realistic.")
print("Using:", DATA_PATH)

## Step 2-3: Load and explore the data

In [ ]:
df = load_dataset(DATA_PATH)
print("Columns:", list(df.columns), "| rows:", len(df))
df.head(8)

In [ ]:
print("Missing values:", int(df["message"].isna().sum()))
print("Duplicate messages:", int(df.duplicated(subset="message").sum()))
df = df.drop_duplicates(subset="message").dropna(subset=["message"]).reset_index(drop=True)

counts = df["label"].value_counts()
print(counts.to_string())
print(f"Spam share: {100 * counts['spam'] / len(df):.1f}%  -> imbalanced, so accuracy alone is misleading")

ax = sns.countplot(data=df, x="label", order=["ham", "spam"], hue="label",
                   palette=["#4C78A8", "#E45756"], legend=False)
for c in ax.containers:
    ax.bar_label(c)
ax.set_title("Class distribution")
plt.show()

In [ ]:
df["length"] = df["message"].str.len()
print(df.groupby("label")["length"].mean().round(1).rename("avg length (chars)").to_string())
sns.histplot(data=df, x="length", hue="label", bins=30, element="step",
             palette={"ham": "#4C78A8", "spam": "#E45756"})
plt.title("Message length by class"); plt.show()

## Step 4: Preprocess the text
Lowercase, remove punctuation and numbers, remove stop words (keeping a few spam-signal words such as *you*, *your*,
*now*), and map labels to numbers (ham = 0, spam = 1).

In [ ]:
df = prepare(df)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)   # drop messages that are empty after cleaning
pd.set_option("display.max_colwidth", 90)
df[["message", "clean", "target"]].sample(6, random_state=1)

In [ ]:
for label in ["ham", "spam"]:
    words = " ".join(df.loc[df.label == label, "clean"]).split()
    print(label.upper(), "->", pd.Series(words).value_counts().head(12).to_dict())

## Step 5: Split the data
Stratified 80/20 split so both sets keep the same spam ratio. The test set is only used for the final evaluation.

In [ ]:
train, test = train_test_split(df, test_size=0.2, stratify=df["target"], random_state=RANDOM_STATE)
y_train, y_test = train["target"].values, test["target"].values
print("train:", len(train), "test:", len(test))
print("spam ratio train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

## Step 6: Convert text to features (TF-IDF)
The vectorizer is fitted on the training data only, then applied to the test data (no data leakage).

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train = vec.fit_transform(train["clean"])   # fit on TRAIN only
X_test = vec.transform(test["clean"])         # only transform the test data
print("Feature matrix:", X_train.shape, "| vocabulary size:", len(vec.vocabulary_))

## Step 7: Tune and train the models
Each model's main setting is chosen with 5-fold cross-validation on the **training set only**, scored on spam F1.
The final model is picked by its cross-validation score, so the test set stays untouched until Step 8.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
candidates = {
    "Multinomial Naive Bayes": (MultinomialNB(), {"alpha": [1.0, 0.3, 0.1, 0.03]}),
    "Logistic Regression": (LogisticRegression(max_iter=2000, class_weight="balanced",
                                               random_state=RANDOM_STATE), {"C": [1, 3, 10]}),
    "Linear SVM": (LinearSVC(class_weight="balanced", random_state=RANDOM_STATE), {"C": [0.1, 0.3, 1]}),
}
tuned = {}
for name, (model, grid) in candidates.items():
    gs = GridSearchCV(model, grid, scoring="f1", cv=cv).fit(X_train, y_train)
    tuned[name] = gs
    print(f"{name:26s} best params {gs.best_params_}   CV F1 = {gs.best_score_:.3f}")

best = max(tuned, key=lambda n: tuned[n].best_score_)
print("\nSelected by cross-validation:", best)

## Step 8: Evaluate on unseen test messages

In [ ]:
preds, rows = {}, []
for name, gs in tuned.items():
    preds[name] = gs.predict(X_test)
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, preds[name]),
                 "precision": precision_score(y_test, preds[name], zero_division=0),
                 "recall": recall_score(y_test, preds[name], zero_division=0),
                 "f1": f1_score(y_test, preds[name], zero_division=0)})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
print("Selected model:", best)
print(classification_report(y_test, preds[best], target_names=["ham", "spam"], digits=3))
cm = confusion_matrix(y_test, preds[best])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {best}"); plt.show()

## Step 9: Analyze the errors

In [ ]:
errors = test.assign(pred=preds[best])
fp = errors[(errors.target == 0) & (errors.pred == 1)]
fn = errors[(errors.target == 1) & (errors.pred == 0)]
print(f"False positives (real message marked spam): {len(fp)}")
for m in fp["message"].head(10):
    print("  -", m)
print(f"\nFalse negatives (spam missed): {len(fn)}")
for m in fn["message"].head(10):
    print("  -", m)

Read the lists above and look for patterns: very short messages, spam written like a personal text, premium-rate
keywords with a short code, unusual spelling, mixed language. Blocking a real message (false positive) is usually
worse than letting some spam through, so watch precision as well as recall.

## Optional: words that indicate spam

In [ ]:
lr = tuned["Logistic Regression"].best_estimator_
names = np.array(vec.get_feature_names_out())
order = np.argsort(lr.coef_[0])[::-1][:15]
plt.barh(names[order][::-1], lr.coef_[0][order][::-1], color="#E45756")
plt.title("Top spam words / phrases (Logistic Regression)"); plt.tight_layout(); plt.show()

## Step 11: Test with new messages
Uses the model saved by `python src/train_model.py --data data/spam.csv` (run it once from the project root first).

In [ ]:
from predict import predict_message
tests = [
    "You won a free prize! Click now to claim.",
    "Are we meeting at 5 pm today?",
    "Urgent! Your account is blocked. Verify your details immediately.",
    "Your order has shipped and will arrive tomorrow.",   # genuine but promotional-looking
    "ok",                                                 # very short
    "",                                                   # empty input
]
try:
    for t in tests:
        r = predict_message(t)
        print(f"{t!r:70s} ->", r["message"] if not r["ok"] else f"{r['label']} ({r['spam_probability']:.2f})")
except FileNotFoundError as e:
    print(e)

## Conclusions
* Spam is a minority class, so precision, recall and F1 on the spam class matter more than accuracy.
* Tuning the model settings with cross-validation helped more than changing the model type.
* The remaining mistakes are mostly spam that looks like normal text and very short premium-rate messages.
  More varied training data would help more than further tuning.
* The model was trained on one dataset (UK SMS from the 2000s), so other regions and newer spam styles may be missed.